# Compact ONNX Artifact View

This notebook publishes a compact NeuroGolf ONNX submission artifact with an audit-friendly path from a scored archive to `submission.zip`. The high-level approach is a conservative taskwise portfolio: start from a validated ONNX set, compare candidate task graphs in isolation, and only keep replacements that preserve the 400-file contract while reducing graph cost.

The attached dataset contains the exact artifact receipt for a **6424.02** public leaderboard version. This refresh includes a compact task-specific convolution graft selected by an A/B audit while keeping the public notebook focused on deterministic reconstruction, manifest checks, and reproducibility.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import zipfile

import pandas as pd

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
PUBLIC_SCORE = 7113.63
SUBMISSION_DESCRIPTION = 'ngf top1 A sajayr neurogolf 7015.36 top2 B kojimar audited overrides 6507.91 AB audit v2'
PRIVATE_SOURCE = 'franksunp/ngf-sajayr701536-kojimar650791-ab-audit-private'

source_dirs = sorted(
    p for p in INPUT.rglob('*')
    if p.is_dir() and ((p / 'submission.zip').exists() or (p / 'ab_audit_manifest.json').exists())
)
assert source_dirs, 'Attach the scored private A/B audit notebook output source.'
ASSET = source_dirs[0]

manifest_path = ASSET / 'ab_audit_manifest.json'
if manifest_path.exists():
    summary = json.loads(manifest_path.read_text())
else:
    summary = {}

picks_path = ASSET / 'ab_audit_picks.csv'
if picks_path.exists():
    picks = pd.read_csv(picks_path)
else:
    picks = pd.DataFrame()

print('asset:', ASSET)
print('private source:', PRIVATE_SOURCE)
print('competition:', 'neurogolf-2026')
print('public LB:', PUBLIC_SCORE)
print('submission description:', SUBMISSION_DESCRIPTION)
print('manifest run_id:', summary.get('run_id'))
print('source picks:', summary.get('picks'))


## Build submission.zip

Kaggle datasets can expose a zip either as a file, preserved binary bytes, or an expanded directory. The cell below handles those layouts and writes the submission artifact expected by the competition.


In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def write_zip_from_task_dir(task_dir: Path, out_zip: Path) -> None:
    task_files = sorted(task_dir.glob('task*.onnx'))
    assert len(task_files) == 400, f'expected 400 ONNX files, found {len(task_files)}'
    names = [p.name for p in task_files]
    expected = [f'task{i:03d}.onnx' for i in range(1, 401)]
    assert names == expected, 'task filenames are not task001.onnx ... task400.onnx'
    with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in task_files:
            zf.write(p, arcname=p.name)


out_zip = WORK / 'submission.zip'
source_zip = ASSET / 'submission.zip'
task_dir = ASSET / 'selected_submission'

if source_zip.exists():
    shutil.copy2(source_zip, out_zip)
    print('copied scored notebook output submission.zip')
else:
    assert task_dir.exists(), 'source must contain submission.zip or selected_submission/task*.onnx'
    write_zip_from_task_dir(task_dir, out_zip)
    print('rebuilt zip from expanded selected_submission directory')

with zipfile.ZipFile(out_zip) as zf:
    names = sorted(zf.namelist())
assert names == [f'task{i:03d}.onnx' for i in range(1, 401)]

rebuilt_sha = sha256_file(out_zip)
manifest_sha = summary.get('submission_sha256')
print('wrote:', out_zip)
print('zip size bytes:', out_zip.stat().st_size)
print('submission.zip sha256:', rebuilt_sha)
print('manifest sha256:', manifest_sha)
print('scored public LB:', PUBLIC_SCORE)
if manifest_sha:
    assert rebuilt_sha == manifest_sha


In [ ]:
if not picks.empty:
    view = picks.copy()
    view['winner_cost'] = pd.to_numeric(view['winner_cost'], errors='coerce')
    display_cols = [
        'task_id', 'winner', 'winner_cost', 'a_cost', 'b_cost',
        'a_status', 'b_status', 'a_fail', 'b_fail'
    ]
    view[display_cols].sort_values('winner_cost', ascending=False).head(12)
else:
    pd.DataFrame([{
        'source': PRIVATE_SOURCE,
        'public_lb': PUBLIC_SCORE,
        'submission_sha256': summary.get('submission_sha256'),
    }])


The output file `/kaggle/working/submission.zip` is copied from preserved scored archive bytes and checked against the recorded SHA256. The receipt, SHA check, and per-task manifest make the artifact easy to audit without requiring expensive graph search inside the public version.

The key idea is intentionally simple: use strict artifact accounting around a compact ONNX portfolio, keep risky exploration out of the final public path, and publish a deterministic notebook that rebuilds exactly the scored submission artifact.
